# 01 · Descenso del gradiente para regresión múltiple, desde cero

**Módulo 3 · Sesión 6** — Regresión lineal

## Objetivos

El notebook `04-gradientes-intuicion.ipynb` del módulo 1 construyó el descenso del
gradiente paso a paso —1D, 2D, regla de la cadena— y terminó ajustando una regresión
**simple** (`promedio_anterior → nota_final`) sobre `rendimiento-estudiantes.csv`,
comparando contra la solución exacta de mínimos cuadrados. Este notebook **parte de ahí**:

1. Extiende ese mismo ajuste a **regresión múltiple**, en forma vectorizada
   (matriz de diseño completa, no una derivada a la vez).
2. Implementa las tres variantes — **batch, mini-batch y SGD** — y compara su convergencia.
3. Muestra en código, no solo en teoría, por qué **escalar los predictores** antes de
   descender es indispensable.

No se repite qué es un gradiente ni la regla de la cadena: ver `05-calculo-y-probabilidad.md`
y el notebook 04 del módulo 1 si hace falta repasarlo.

**Paquetes:** `numpy`, `pandas`, `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SEMILLA = 42
rng = np.random.default_rng(SEMILLA)

## 1. Los datos: los mismos del módulo 1, con más predictores

El notebook 04 del módulo 1 solo usó `promedio_anterior`. Aquí se agregan
`horas_estudio_semana`, `asistencia_pct`, `edad`, `estrato` y `trabaja` — mismo dataset,
mismo objetivo (`nota_final`), ahora con $p=6$ predictores en vez de 1.

In [ ]:
datos = pd.read_csv("../datos/rendimiento-estudiantes.csv")

columnas_x = [
    "promedio_anterior",
    "horas_estudio_semana",
    "asistencia_pct",
    "edad",
    "estrato",
    "trabaja",
]
X_crudo = datos[columnas_x].to_numpy(dtype=float)
y = datos["nota_final"].to_numpy(dtype=float)

print(f"X: {X_crudo.shape}  (n={X_crudo.shape[0]}, p={X_crudo.shape[1]})")
datos[columnas_x + ["nota_final"]].describe().round(2)

Las escalas ya son muy distintas a simple vista: `asistencia_pct` va de 0 a 100,
`trabaja` es 0/1, `edad` ronda los 20-30. Volveremos sobre esto en la sección 4.

## 2. Forma vectorizada: costo y gradiente

Con la matriz de diseño $\mathbf{X}$ (le añadimos una columna de unos para el intercepto) y
$\boldsymbol{\beta} \in \mathbb{R}^{p+1}$:

$$
\mathcal{L}(\boldsymbol{\beta}) = \frac{1}{n} \lVert \mathbf{y} - \mathbf{X}\boldsymbol{\beta}
\rVert_2^2 \qquad\qquad \nabla_{\boldsymbol{\beta}} \mathcal{L} = -\frac{2}{n}
\mathbf{X}^{\top}(\mathbf{y} - \mathbf{X}\boldsymbol{\beta})
$$

Cada una de estas dos funciones reemplaza los `derivada_analitica` escritos a mano del
notebook 04 — la vectorización es lo único nuevo.

In [ ]:
def anadir_intercepto(X):
    unos = np.ones((X.shape[0], 1))
    return np.hstack([unos, X])


def costo(beta, X, y):
    residuales = y - X @ beta
    return np.mean(residuales**2)


def gradiente(beta, X, y):
    residuales = y - X @ beta
    return -2 * X.T @ residuales / len(y)

## 3. Estandarizar antes de descender

Estandarizamos cada predictor ($z = (x-\bar{x})/s$) **antes** de correr el descenso — no
después. Guardamos media y desviación para poder interpretar o revertir más adelante.

In [ ]:
medias = X_crudo.mean(axis=0)
desvios = X_crudo.std(axis=0)
X_z = (X_crudo - medias) / desvios
X_disenio = anadir_intercepto(X_z)

## 4. Descenso batch, y su costo con y sin escalado

Primero, la versión correcta: descenso batch sobre los datos **estandarizados**.

In [ ]:
def descenso_batch(X, y, tasa, pasos):
    beta = np.zeros(X.shape[1])
    historial = [costo(beta, X, y)]
    for _ in range(pasos):
        beta = beta - tasa * gradiente(beta, X, y)
        historial.append(costo(beta, X, y))
    return beta, np.array(historial)


beta_batch, historial_batch = descenso_batch(X_disenio, y, tasa=0.1, pasos=300)
print(f"Costo final (estandarizado): {historial_batch[-1]:.5f}")

Ahora la misma función, misma tasa de aprendizaje, pero sobre los datos **sin
estandarizar** — solo con el intercepto añadido. Es el error más común al implementar
descenso del gradiente por primera vez.

In [ ]:
X_sin_escalar = anadir_intercepto(X_crudo)
with np.errstate(over="ignore", invalid="ignore"):  # el desborde es el punto del ejemplo
    beta_sin_escalar, historial_sin_escalar = descenso_batch(
        X_sin_escalar, y, tasa=0.1, pasos=300
    )

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].plot(historial_batch)
ejes[0].set_title("Estandarizado: converge")
ejes[0].set_xlabel("Paso")
ejes[0].set_ylabel("MSE")

ejes[1].plot(historial_sin_escalar)
ejes[1].set_yscale("log")
ejes[1].set_title("Sin estandarizar: diverge (escala log)")
ejes[1].set_xlabel("Paso")
plt.tight_layout()
plt.show()

print(f"Costo en el paso 5 sin escalar: {historial_sin_escalar[5]:.2e}")
print(f"Último costo sin escalar: {historial_sin_escalar[-1]} (se desborda a infinito)")

Con la **misma** tasa de aprendizaje que converge sin problema sobre datos estandarizados,
la versión sin escalar diverge: el costo explota en las primeras iteraciones. No es que el
algoritmo esté mal escrito — es que la superficie de costo, sin escalar, es un valle tan
alargado que ese $\eta$ es demasiado grande en la dirección corta del valle. Para que
convergiera sin escalar habría que reducir $\eta$ varios órdenes de magnitud, y aun así
tardaría muchas más iteraciones. De aquí en adelante, todo el notebook trabaja sobre
`X_disenio` (estandarizado).

## 5. Batch vs. mini-batch vs. SGD

Las tres variantes comparten `gradiente`; solo cambia sobre cuántas filas se calcula en
cada paso.

In [ ]:
def descenso_estocastico(X, y, tasa, epocas, tamano_lote, rng):
    n = X.shape[0]
    beta = np.zeros(X.shape[1])
    historial = [costo(beta, X, y)]
    for _ in range(epocas):
        orden = rng.permutation(n)
        for inicio in range(0, n, tamano_lote):
            idx = orden[inicio : inicio + tamano_lote]
            beta = beta - tasa * gradiente(beta, X[idx], y[idx])
        historial.append(costo(beta, X, y))  # costo sobre todos los datos, al cerrar la época
    return beta, np.array(historial)


EPOCAS = 60
beta_mb, historial_mb = descenso_estocastico(
    X_disenio, y, tasa=0.1, epocas=EPOCAS, tamano_lote=32, rng=np.random.default_rng(SEMILLA)
)
beta_sgd, historial_sgd = descenso_estocastico(
    X_disenio, y, tasa=0.1, epocas=EPOCAS, tamano_lote=1, rng=np.random.default_rng(SEMILLA)
)

# Igualamos el eje x de batch a "épocas" (aquí, 1 paso = 1 época) para comparar de forma justa.
pasos_por_epoca_batch = max(len(historial_batch) // EPOCAS, 1)
historial_batch_por_epoca = historial_batch[::pasos_por_epoca_batch][: EPOCAS + 1]

plt.figure(figsize=(7, 4.5))
plt.plot(historial_batch_por_epoca, label="Batch (n=400)")
plt.plot(historial_mb, label="Mini-batch (b=32)")
plt.plot(historial_sgd, label="SGD (b=1)")
plt.yscale("log")
plt.xlabel("Época")
plt.ylabel("MSE (escala log)")
plt.title("Convergencia por variante, misma tasa de aprendizaje")
plt.legend()
plt.show()

Lectura de la curva: batch baja de forma suave y monótona — cada paso usa el gradiente
exacto. SGD baja rápido al principio (muchas actualizaciones por época) pero se queda
oscilando cerca del mínimo sin asentarse del todo, porque cada paso usa el gradiente
ruidoso de una sola fila. Mini-batch queda en un punto intermedio: menos ruido que SGD, más
actualizaciones por época que batch.

## 6. ¿Llegaron al mismo sitio que la ecuación normal?

`01-regresion-lineal.md` dedujo la solución exacta $\boldsymbol{\beta} =
(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$. Si el descenso está bien
implementado, batch debe converger a (casi) ese mismo vector.

In [ ]:
beta_exacto = np.linalg.solve(X_disenio.T @ X_disenio, X_disenio.T @ y)

comparacion = pd.DataFrame(
    {
        "variable": ["intercepto"] + columnas_x,
        "ecuacion_normal": beta_exacto.round(4),
        "descenso_batch": beta_batch.round(4),
        "descenso_mini_batch": beta_mb.round(4),
        "descenso_sgd": beta_sgd.round(4),
    }
)
comparacion

Batch y mini-batch coinciden con la ecuación normal hasta el tercer o cuarto decimal. SGD
se queda visiblemente más lejos con el mismo número de épocas — es el precio del ruido que
se vio en la curva de aprendizaje; con más épocas o una tasa decreciente, también convergería.

## Resumen

| Lo que se hizo | Conecta con |
|---|---|
| Costo y gradiente vectorizados para $p$ predictores | `01-regresion-lineal.md`, ecuación normal |
| Divergencia sin escalar, con la misma $\eta$ que converge escalado | `02-descenso-gradiente.md`, sección 4 |
| Batch vs. mini-batch vs. SGD sobre la misma curva de aprendizaje | Entrenamiento de redes neuronales (módulo 5) |
| Coeficientes del descenso vs. la solución exacta | Confirma que la implementación es correcta antes de usarla en datos reales (notebook 02) |